In [12]:
import time
import logging

logging.basicConfig(level=logging.INFO)

def update_order_status(order_id: str, status: str, error_message: str = None):
    """
    Эмуляция PATCH/PUT-запроса к /api/archive-orders/{id} для обновления статуса
    """
    payload = {"status": status}
    if error_message:
        payload["error"] = error_message
    logging.info(f"[API UPDATE] Заказ {order_id} -> Статус: {status}")


def process_geo_data_task(order_id: str, input_image_path: str):
    """
    Фоновая функция обработки (Background Task)
    """
    try:
        # 1. Меняем статус на "processing"
        update_order_status(order_id, "processing")
       
        logging.info(f"Начало скачивания и обработки файла: {input_image_path}")
       
        # Эмуляция долгой геообработки (SNAP / Python / GDAL)
        time.sleep(2)
       
        # Симуляция проверки: если файла нет — бросаем ошибку
        if not input_image_path.endswith((".tif", ".SAFE")):
            raise ValueError("Неподдерживаемый формат исходного снимка")

        # 2. Успешное завершение -> сохранение результатов в /app/results
        logging.info("Генерация маски завершена. Файлы записаны в /app/results")
       
        # 3. Переводим статус в "delivered"
        update_order_status(order_id, "delivered")

    except Exception as e:
        logging.error(f"Ошибка при обработке заказа {order_id}: {str(e)}")
        # 4. При сбое переводим статус в "failed"
        update_order_status(order_id, "failed", error_message=str(e))

# --- Проверка работы ---
# Тест 1: Успешный сценарий
process_geo_data_task(order_id="order_abc_123", input_image_path="/app/data/scene.tif")

# Тест 2: Сценарий с ошибкой
process_geo_data_task(order_id="order_xyz_999", input_image_path="/app/data/corrupted_file.txt")

INFO:root:[API UPDATE] Заказ order_abc_123 -> Статус: processing
INFO:root:Начало скачивания и обработки файла: /app/data/scene.tif
INFO:root:Генерация маски завершена. Файлы записаны в /app/results
INFO:root:[API UPDATE] Заказ order_abc_123 -> Статус: delivered
INFO:root:[API UPDATE] Заказ order_xyz_999 -> Статус: processing
INFO:root:Начало скачивания и обработки файла: /app/data/corrupted_file.txt
ERROR:root:Ошибка при обработке заказа order_xyz_999: Неподдерживаемый формат исходного снимка
INFO:root:[API UPDATE] Заказ order_xyz_999 -> Статус: failed


In [11]:
import json

with open("3_raw_polygons.json", "r", encoding="utf-8") as file:
    data = json.load(file)


geojson_result = {"type": "FeatureCollection", "features": []}

for item in data:
    correct_coords = [[lon, lat] for lat, lon in item["coordinates_lat_lon"]]

    feature = {"type": "feature",
               "geometry": {"type": "Polygon",
                            "coordinates": [correct_coords]
                            },
                "properties": {"item_id": item["id"],
                               "area_sqkm": float(item["area_sqkm"]),
                                "product_type": "WATER_MASK"
                               }
               }
    geojson_result["features"].append(feature)

with open ("correct_geojson.geojson", "w", encoding="utf-8") as file:
    json.dump(geojson_result, file, indent=4, ensure_ascii=False)